In [ ]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append("..")
from modeling import COVARIATES

DATA_DIR = Path("/home/ec2-user/studies/Haochen_food/code_micro/data")
OUT_DIR = DATA_DIR / "train_pathways"
PREVALENCE_MIN_COUNT = 200
META = ["participant_id", "cohort", "research_stage", "array_index", "collection_date"]


def clean_ids(s):
    return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.replace(r"\s+", "", regex=True)


def sanitize(columns):
    names = [re.sub(r"_+", "_", re.sub(r"[^0-9a-zA-Z_]", "_", str(c)).strip("_")) for c in columns]
    return [f"{n}_{names[:i].count(n)}" if n in names[:i] else n for i, n in enumerate(names)]


def to_date(values):
    return pd.to_datetime(values, errors="coerce", utc=True).dt.tz_convert(None).dt.normalize()


diet = pd.read_csv(DATA_DIR / "diet_experiment" / "diet_feature_matrix_baseline_full.csv", low_memory=False)
covariates = pd.read_csv(DATA_DIR / "covariates" / "covariates_age_sex_bmi_smoking_oralhealth_baseline.csv")
pathways = pd.read_csv(DATA_DIR / "csv_oral" / "pathway_abundance_pathway_level.csv", low_memory=False)
for df in (diet, covariates, pathways):
    df["participant_id"] = clean_ids(df["participant_id"])
pathways.columns = sanitize(pathways.columns)

baseline = pathways["research_stage"].astype(str).str.strip().eq("00_00_visit")
if baseline.any():
    pathways = pathways[baseline]
else:
    cohort = pd.read_csv(DATA_DIR / "csv_diet" / "select_diet_oral.csv")
    oral_date = cohort.assign(participant_id=clean_ids(cohort["participant_id"])).set_index("participant_id")["oral_collection_date"]
    pathways = pathways[to_date(pathways["collection_date"]).eq(pathways["participant_id"].map(to_date(oral_date)))]
pathways = pathways.assign(_date=to_date(pathways["collection_date"])).sort_values(["participant_id", "_date"])
pathways = pathways.drop(columns="_date").drop_duplicates("participant_id")

In [ ]:
pathway_cols = [c for c in pathways.columns if c not in META and not re.search("UNMAPPED|UNINTEGRATED", c, re.IGNORECASE)]
common = set(diet["participant_id"]) & set(covariates["participant_id"]) & set(pathways["participant_id"])
pathways = pathways[pathways["participant_id"].isin(common)].reset_index(drop=True)
abundance = pathways[pathway_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
prevalence_n = (abundance > 0).sum()
retained = prevalence_n[prevalence_n >= PREVALENCE_MIN_COUNT].index.tolist()
values = abundance[retained].astype(float)
for col in retained:
    values.loc[values[col] <= 0, col] = 0.5 * np.percentile(values.loc[values[col] > 0, col], 1)
pathway_log10 = pd.concat([pathways[["participant_id"]], np.log10(values.div(values.sum(axis=1), axis=0) * 1e6)], axis=1)

OUT_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"pathway": pathway_cols, "prevalence_n": prevalence_n.to_numpy(), "prevalence": (abundance > 0).mean().to_numpy(),
              "kept_for_modeling": prevalence_n.ge(PREVALENCE_MIN_COUNT).to_numpy()}).to_csv(
    OUT_DIR / "functional_pathway_summary_baseline.csv", index=False)
model_input = (covariates[covariates["participant_id"].isin(common)][["participant_id"] + COVARIATES]
               .merge(diet, on="participant_id").merge(pathway_log10, on="participant_id"))
model_input.to_csv(OUT_DIR / "model_input_diet_cov_functional_pathways_baseline.csv", index=False)
print("participants:", len(model_input), "| retained pathways:", len(retained), "of", len(pathway_cols))